In [26]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import html
import unicodedata

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

In [27]:
class SpellChecker:
    def __init__(self, word_freqs):
        self.WORDS = word_freqs
        self.N = sum(word_freqs.values())

    def P(self, word):
        return self.WORDS.get(word, 0) / self.N

    def correction(self, word):
        return max(self.candidates(word), key=self.P)

    def candidates(self, word):
        return (self.known([word]) or self.known(self.edits1(word)) or self.known(self.edits2(word)) or [word])

    def known(self, words):
        return set(w for w in words if w in self.WORDS)

    def edits1(self, word):
        letters    = 'abcdefghijklmnopqrstuvwxyz'
        splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
        deletes    = [L + R[1:]               for L, R in splits if R]
        transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
        replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
        inserts    = [L + c + R               for L, R in splits for c in letters]
        return set(deletes + transposes + replaces + inserts)

    def edits2(self, word):
        return (e2 for e1 in self.edits1(word) for e2 in self.edits1(e1))

In [28]:
df = pd.read_csv('../data/raw/recipes.csv')

df["Name"] = df["Name"].astype(str).apply(html.unescape)
df["Description"] = df["Description"].astype(str).apply(html.unescape)
df["RecipeIngredientParts"] = df["RecipeIngredientParts"].astype(str).apply(html.unescape)
df["RecipeInstructions"] = df["RecipeInstructions"].astype(str).apply(html.unescape)

df['SearchCorpus'] = df['Name'] + ' ' + df['RecipeIngredientParts'] + ' ' + df['RecipeInstructions']

In [29]:
def spell_preprocessor(s):
    s = str(s).lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')
    s = re.sub(r'[^a-z\s]', ' ', s)
    return s

In [39]:
spell_vectorizer = CountVectorizer(preprocessor=spell_preprocessor, min_df=100)
spell_counts = spell_vectorizer.fit_transform(df['SearchCorpus'])

In [40]:
vocab_counts = spell_counts.sum(axis=0).A1
vocab_words = spell_vectorizer.get_feature_names_out()
word_frequencies = dict(zip(vocab_words, [int(c) for c in vocab_counts]))

In [41]:
spell_checker = SpellChecker(word_frequencies)

with open('../resources/spell_checker.pkl', 'wb') as f:
    pickle.dump(spell_checker, f)

In [42]:
with open('../resources/spell_checker.pkl', 'rb') as f:
    spell_checker = pickle.load(f)

query = "eeg satt papper heet"
query_words = query.split()
    
corrected_words = []
has_typo = False

for word in query_words:
    corrected = spell_checker.correction(word)
    if corrected != word:
        has_typo = True
    corrected_words.append(corrected)

suggested_query = " ".join(corrected_words)
print(suggested_query)

egg salt pepper heat
